<p class="h1">ECE 447 - Notebook 08</p>
<p class="h2">MLOps</p>

In [ ]:
# pip install mlflow

In [ ]:
# pip install wandb -qqq

In [ ]:
#run it from the terminal
# !wandb login --relogin

In [ ]:
###### This block is for MLflow

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet, LinearRegression
from urllib.parse import urlparse

import mlflow
import mlflow.sklearn

from datetime import datetime
import logging

logging.basicConfig(level=logging.WARN)
logger = logging.getLogger(__name__)


def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2


if __name__ == "__main__":
    # data config
    data_name = "wine"

    # model config
    # model_name = "LinearRegression"
    model_name = "ElasticNet"
    if model_name == "ElasticNet":
        alpha =  0.75
        l1_ratio =  0.0

    random_state = 39

    run_name = datetime.now().strftime("%Y%m%d%H%M%S")
    experiment_name = "{}_{}".format(model_name, run_name)

    ################################################
    ################################################

    # Read the wine-quality csv file from the URL
    csv_url = (
        "http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    )
    try:
        data = pd.read_csv(csv_url, sep=";")
    except Exception as e:
        logger.exception(
            "Unable to download training & test CSV, check your internet connection. Error: %s", e
        )

    # Split the data into training and test sets. (0.75, 0.25) split.
    train, test = train_test_split(data)

    # The predicted column is "quality" which is a scalar from [3, 9]
    train_x = train.drop(["quality"], axis=1)
    test_x = test.drop(["quality"], axis=1)
    train_y = train[["quality"]]
    test_y = test[["quality"]]

    mlflow.set_experiment(experiment_name)
    
    with mlflow.start_run(run_name=run_name):
        if model_name == "ElasticNet":
            model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio)
        elif model_name == "LinearRegression":
            model = LinearRegression()

        model.fit(train_x, train_y)
        
        predicted_qualities = model.predict(test_x)

        (rmse, mae, r2) = eval_metrics(test_y, predicted_qualities)

        if model_name == "ElasticNet":
            print("Elasticnet model (alpha=%f, l1_ratio=%f):" % (alpha, l1_ratio))
        elif model_name == "LinearRegression":
             print("Linear Regression")
        print("  RMSE: %s" % rmse)
        print("  MAE: %s" % mae)
        print("  R2: %s" % r2)

        mlflow.log_param("alpha", alpha)
        mlflow.log_param("l1_ratio", l1_ratio)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("mae", mae)

        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        # Model registry does not work with file store
        if tracking_url_type_store != "file":

            # Register the model
            # There are other ways to use the Model Registry, which depends on the use case,
            # please refer to the doc for more information:
            # https://mlflow.org/docs/latest/model-registry.html#api-workflow
            mlflow.sklearn.log_model(model, "model", registered_model_name="{}_{}".format(model_name, data_name))
        else:
            mlflow.sklearn.log_model(model, "model")

In [ ]:
!mlflow ui

In [ ]:
###### This block is for wandb

import wandb
import random
from datetime import datetime

run_name = datetime.now().strftime("%Y%m%d%H%M%S")

# start a new wandb run to track this script
wandb.init(
    # set the wandb project where this run will be logged
    project="project-ece447",
    name = "exp-{}".format(run_name),
    
    # track hyperparameters and run metadata
    config={
    "learning_rate": 0.035,
    "architecture": "CNN",
    "dataset": "MNIST",
    "epochs": 15,
    }
)

artifact = wandb.Artifact(name = "mnist", type="dataset", description = "This is a test")
wandb.log_artifact(artifact)


# simulate training
# this is a random loop to show how wandb works
# this is not a real training.
epochs = 10
offset = random.random() / 9
for epoch in range(2, epochs):
    acc = 1 - 2.2 ** -epoch - random.random() / epoch - offset
    loss = 2 ** -epoch + random.random() / epoch + offset
    
    # log metrics to wandb
    wandb.log({"acc": acc, "loss": loss})
    
# [optional] finish the wandb run, necessary in notebooks
wandb.finish()